# **Day 61: Type Hints**

**Type hints** are small notes in the code that tell **what type of value** a variable or function expects and returns.

```python
def fare(km: float) -> float:
```
This means: `km` should be a float, and the function gives back a float.

**Important:** Python itself **does not check** type hints while running. They are for:
- **People** reading the code (the code explains itself).
- **Code editors** like VS Code, which can warn us about mistakes before we run the code.
- **Tools** like `mypy`, which check the whole project for type mistakes.

In [1]:
def fare(km: float, per_km: float = 20.0) -> float:
    return km * per_km

print(fare(5))
print(fare("5", 2))       # No error! Python does not check type hints while running

100.0
55


The second call gave `"55"` (text repeated), not an error. A tool like `mypy` would have warned us about this mistake.

## **Basic Type Hints**

In [2]:
name: str = "Siliguri"
pin: int = 734001
lat: float = 26.72
is_hill: bool = False
nothing: None = None

## **Type Hints for Collections**

In [3]:
districts: list[str] = ["Kolkata", "Howrah"]
coordinate: tuple[float, float] = (22.57, 88.36)
pins: dict[str, int] = {"Kolkata": 700001}
unique_fish: set[str] = {"ilish", "rui"}

## **Optional Values: X | None**
`int | None` means "an int **or** None". This is used when a function may not find an answer.

In [4]:
def find_pin(city: str) -> int | None:
    return {"Kolkata": 700001, "Midnapore": 721101}.get(city)

print(find_pin("Midnapore"), find_pin("Patna"))

721101 None


## **Example from Remote Sensing: Type Hints Make Functions Clear**
With type hints, anyone can understand what goes in and what comes out, without reading the whole function.

In [5]:
BBox = tuple[float, float, float, float]          # A "type alias": a short name for a type

def ndvi(nir: float, red: float) -> float | None:
    """Return NDVI, or None if it cannot be calculated."""
    if nir + red == 0:
        return None
    return (nir - red) / (nir + red)

def bbox_of(points: list[tuple[float, float]]) -> BBox:
    """Return (min_lon, min_lat, max_lon, max_lat) of (lat, lon) points."""
    lats = [p[0] for p in points]
    lons = [p[1] for p in points]
    return min(lons), min(lats), max(lons), max(lats)

def pixels_to_hectare(count: int, resolution_m: float = 10) -> float:
    return count * resolution_m ** 2 / 10_000

print(ndvi(0.45, 0.05), ndvi(0, 0))
print(bbox_of([(22.57, 88.36), (26.72, 88.39), (21.62, 87.50)]))
print(pixels_to_hectare(12500), "hectare")

0.8 None
(87.5, 21.62, 88.39, 26.72)
125.0 hectare


- `BBox = tuple[float, float, float, float]`: Instead of writing this long type again and again, we give it a short name.
- `-> float | None`: The function returns a number, or `None` when both values are zero.

## **Callable, Any and Literal**

In [6]:
from typing import Callable, Any, Literal

def apply_to_all(func: Callable[[float], float], values: list[float]) -> list[float]:
    return [func(v) for v in values]

def log(x: Any) -> None:            # Any = any type is fine
    print("Log:", x)

Satellite = Literal["Landsat 8", "Sentinel-2"]     # Only these exact values are allowed

def resolution(sat: Satellite) -> int:
    return 30 if sat == "Landsat 8" else 10

print(apply_to_all(abs, [-1.5, 2.0, -3.2]))
log({"any": "thing"})
print(resolution("Sentinel-2"))

[1.5, 2.0, 3.2]
Log: {'any': 'thing'}
10


## **TypedDict: Describing a Dictionary's Keys**

In [7]:
from typing import TypedDict

class SceneInfo(TypedDict):
    scene_id: str
    cloud: float
    date: str

scene: SceneInfo = {"scene_id": "S2A_45QXF_20240105", "cloud": 3.1, "date": "2024-01-05"}
print(scene["cloud"])

3.1


## **Type Hints in Classes**

In [8]:
from dataclasses import dataclass, field

@dataclass
class Shop:
    name: str
    items: dict[str, float] = field(default_factory=dict)

    def add(self, item: str, price: float) -> "Shop":
        self.items[item] = price
        return self

    def cheapest(self) -> tuple[str, float] | None:
        if not self.items:
            return None
        return min(self.items.items(), key=lambda kv: kv[1])

s = Shop("Balaram Mullick").add("Sandesh", 25).add("Rosogolla", 15)
print(s.cheapest())

('Rosogolla', 15)


## **Reading Type Hints**

In [9]:
print(fare.__annotations__)

{'km': <class 'float'>, 'per_km': <class 'float'>, 'return': <class 'float'>}


## **Checking with mypy**
Install it with `pip install mypy` and run `mypy my_file.py` in the terminal. For the `fare("5", 2)` mistake above, it would show:

```
error: Argument 1 to "fare" has incompatible type "str"; expected "float"
```

In [10]:
# Quick question
# Will Python show an error for this line? Why?
# def f(x: int) -> int:
#     return x
# print(f("hello"))